<a href="https://colab.research.google.com/github/rafribeiro/fake_detect_eafs2022/blob/main/06_deepfake_detection_prediction_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estudos na base Celeb-DF (deepfake), utilizando escores de similaridade facial DCNN 

Este notebook permite testar um vídeo questionado, tendo outro vídeo autêntico como referência.

São extraídos 50 imagens faciais de cada vídeo e obtido um conjunto de 2500 escores de similaridade. Deste conjunto são obtidas a mediana, o valor máximo e o valor mínimo, que compõem o vetor de features para o modelo de classificação de deepfakes.

O vetor de features é submetido a um modelo pré-treinado na base Celeb-DF v2

Este código é baseado no trabalho publicado em:

[1] P. M. G. I. Reis e R. O. Ribeiro, “A forensic evaluation method for DeepFake detection using DCNN-based facial similarity scores”, Forensic Science International, p. 111747, jun. 2023, doi: 10.1016/j.forsciint.2023.111747.

(C) Paulo Max G. I. Reis, Rafael O. Ribeiro

In [12]:
##!pip uninstall onnxruntime-gpu


^C
ERROR: Operation cancelled by user


In [50]:
import insightface
import cv2
import os
import pickle
import numpy as np
import pandas as pd
import glob
from statistics import median
from IPython.display import display, HTML
#import ffmpeg
import subprocess
import shutil
import sys
display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))
np.set_printoptions(threshold=np.inf, linewidth=np.inf, precision=4)


In [51]:
# Initialize face detection and recognition model
import tempfile
temp_dir = tempfile.mkdtemp()
model_insightface = insightface.app.FaceAnalysis(allowed_modules=['detection','recognition'], root=temp_dir, providers=['CUDAExecutionProvider'])
model_insightface.prepare(ctx_id=0)

download_path: /tmp/tmprkr0a5ha/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 70661.66KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0'}}
model ignore: /tmp/tmprkr0a5ha/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable'

In [55]:
#!chmod +x process_videos.sh
#!./process_videos.sh

In [52]:
# path to questioned and ref. videos
# questPath = "/content/drive/MyDrive/fakedetect/testing_videos/questioned"
# refPath = "/content/drive/MyDrive/fakedetect/testing_videos/references"
# ffmpeg -i input_video.mp4 -vf "scale='if(gt(iw,ih),850,-1)':'if(gt(iw,ih),-1,850)'" -c:v mpeg4 -profile:v simple -pix_fmt yuv420p -r 30 -b:v 800k -an output_video.mp4


questPath = "/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/questioned"
refPath = "/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references"

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [53]:
def extract_faces(video, num=50, dest_dir = None):
    """ Function to detect faces in frames and export each face to a png file

        Parameters:
        video: full path to video file
        num: how many frames to process. Default = 50 frames / video
        dest_dir: where to save faces """

    assert (dest_dir != None)
    if not os.path.exists(dest_dir):
      os.makedirs(dest_dir)
    print('Extracting {} faces from video {}'.format(num, video))
    vs = cv2.VideoCapture(video)
    #time.sleep(1.0)
    frame_number=0
    total_frames = int(vs.get(cv2.CAP_PROP_FRAME_COUNT))
    frames=np.linspace(1, total_frames, num=num, dtype=int) #frames to look for faces
    ipd=[]


    #process frames
    while True:
        # grab the frame from the threaded video stream
        (ret, frame) = vs.read()
        if not ret:
            break
        (h, w) = frame.shape[0:2]
        frame_number += 1
        if frame_number in frames:
            #print('Processing frame {}'.format(frame_number), end='\r')

            #Detect faces in frame. Only consider faces with detection score >= 0.8)
            faces = model_insightface.get(frame)
            for i, face in enumerate(faces):
                (startX, startY, endX, endY) = face.bbox.astype("int")
                width = endX-startX
                height = endY-startY
                # Discard faces smaller than 20 px in each dimension
                if width < 20 or height < 20:
                    continue

                # add margin (100%) on the detected face
                out_width = (endX-startX)*2.0
                out_height = (endY-startY)*2.0
                startX_out = int((startX+endX)/2 - out_width/2)
                endX_out = int((startX+endX)/2 + out_width/2)
                startY_out = int((startY+endY)/2 - out_height/2)
                endY_out = int((startY+endY)/2 + out_height/2)

                #tests if the output bbox coordinates are out of frame limits
                if startX_out < 0:
                    startX_out = 0
                if endX_out > int(w):
                    endX_out = int(w)
                if startY_out < 0:
                    startY_out = 0
                if endY_out > int(h):
                    endY_out = int(h)

                #export the face (with added margin)
                face_crop = frame[startY_out:endY_out, startX_out:endX_out]
                imgPath = os.path.join(dest_dir, os.path.splitext(os.path.basename(video))[0] + "_" + str(frame_number) + "_face_" + str(i) + ".png")
                cv2.imwrite(imgPath,face_crop)
                ipd.append(np.linalg.norm(face.kps[0] - face.kps[1]))
    vs.release()
    return ipd

def extract_features(imgPath):
    bgrImg = cv2.imread(imgPath)
    center = np.array([int(bgrImg.shape[0]/2),int(bgrImg.shape[1]/2)])
    faces = model_insightface.get(bgrImg)
    dist=[]
    rep = np.empty(512)

    # Indicate if no face is detected
    if len(faces) == 0:
        print("No face found on {}".format(imgPath))
        rep[:] = np.nan
        status = "no face"
        return status, rep

    # Compute centroids of faces and distances from certer of image (125, 125)
    for idx, face in enumerate(faces):
        box=face.bbox.astype(int).flatten()
        centroid = np.array([int((box[0]+box[2])/2),int((box[1]+box[3])/2)])
        dist.append(np.linalg.norm(center-centroid))

    # Get embeddings of the face with centroid closest to the center of the image
    idx_face = dist.index(min(dist))
    rep = faces[idx_face].normed_embedding

    status = "ok"
    return status, rep

# extract facial recognition features from each image and returns a dict with
# features from 50 faces
def get_features_from_video(folder):
  imgList = glob.glob(folder+"*.png")
  features_dict = {}
  for imgPath in imgList:
    key = os.path.basename(imgPath)
    features_dict[key] = extract_features(imgPath)
  return features_dict

#Function to generate scores sets
def get_scores(ref, quest):
    scores=[]
    gal_ref = {key:val[1] for key,val in ref.items() if val[0] == "ok"}
    gal_quest = {key:val[1] for key,val in quest.items() if val[0] == "ok"}

    for x in gal_ref.values():
        for y in gal_quest.values():
            sim = np.dot(x,y)/(np.linalg.norm(x)*np.linalg.norm(y))
            scores.append(sim)
    return scores

In [74]:
import os
video_ref_paths = []
for root, dirs, files in os.walk(refPath):
  for file in files:
    video_ref_paths.append(os.path.join(root, file))

print(video_ref_paths)


['/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP2.mp4', '/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP4.mp4', '/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP1.mp4', '/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP5.mp4', '/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP6.mp4']


In [75]:
video_quest_paths = []
for root, dirs, files in os.walk(questPath):
  for file in files:
    video_quest_paths.append(os.path.join(root, file))

print(video_quest_paths)

['/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/questioned/trump.mp4', '/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/questioned/TRUMP3.mp4']


In [76]:
# extract faces from questioned video
quest_dir = questPath+"_imgs/"
ipds_quest=[]
num=50

if os.path.exists(quest_dir) and os.path.isdir(quest_dir):
        shutil.rmtree(quest_dir)
        print(f"{quest_dir}' directory removed")    
for path in video_quest_paths:
    dest_dir= quest_dir+os.path.basename(path)[:-4]+"/"
    questPath2 = path
    extract_faces(questPath2, num, dest_dir=dest_dir)
print("Questioned faces extracted. Please review the extracted faces before continuing.")

/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/questioned_imgs/' directory removed
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/questioned/trump.mp4
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/questioned/TRUMP3.mp4
Questioned faces extracted. Please review the extracted faces before continuing.


In [77]:
# extract faces from reference video
ref_dir = refPath+"_imgs/"
num=50
if os.path.exists(ref_dir) and os.path.isdir(ref_dir):
        shutil.rmtree(ref_dir)
        print(f"{ref_dir}' directory removed")
for path in video_ref_paths:
  refPath2 = path
  extract_faces(refPath2, num, dest_dir = ref_dir)
print("Reference faces extracted. Please review the extracted faces before continuing.")

/home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references_imgs/' directory removed
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP2.mp4
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP4.mp4
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP1.mp4
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP5.mp4
Extracting 50 faces from video /home/jupyter-paulo.pmgir/Curso VE Trilha/Notebooks/UD13 - Deepfakes/references/TRUMP6.mp4
Reference faces extracted. Please review the extracted faces before continuing.


In [78]:
def transform_lr(v,model):
    v = v.reshape(1, -1)
    probs=model.predict_proba(v)
    lr=probs[:,1]/probs[:,0]
    return lr

In [79]:
from tqdm import tqdm
LR=[]
#Load pre-trained model
with open("/srv/data/system_share/deepfake_model/model_LR_max_median_stats.pkl", "rb") as f:
  model_ff = pickle.load(f)
# extract features and compute similarity scores between questioned and reference videos
i=0
for path in tqdm(video_quest_paths):
  dir= quest_dir+os.path.basename(path)[:-4]+"/"
  quest_gallery = get_features_from_video(dir)
  ref_gallery = get_features_from_video(ref_dir)
  scores = get_scores(ref_gallery, quest_gallery) 
  test_features = pd.DataFrame()
  test_features['max'] = [max(scores)]
  test_features['median'] = [median(scores)]  
  print(test_features)
  V= test_features.to_numpy()
  LR.append(transform_lr(V[0], model_ff))
  i+=1


 50%|█████     | 1/2 [00:05<00:05,  5.52s/it]

        max    median
0  0.603461  0.458774


100%|██████████| 2/2 [00:11<00:00,  5.54s/it]

        max    median
0  0.862436  0.647209


In [80]:
i=0
for path in (video_quest_paths):
  print("The evidence in "+ os.path.basename(path)[:-4] + " presents a likelihood ratio of" + f'{LR[i], np.log10(LR[i])}')
  i+=1

The evidence in trump presents a likelihood ratio of(array([0.3952]), array([-0.4032]))
The evidence in TRUMP3 presents a likelihood ratio of(array([339.3594]), array([2.5307]))
